# Embeddings na Prática
### Como texto vira geometria — e por que isso muda tudo em NLP

---

Após a tokenização, cada token recebe um ID numérico.  
Mas IDs não carregam significado — `42` não é mais parecido com `43` do que com `1000`.

**Embeddings** resolvem isso: eles convertem tokens (e frases inteiras) em **vetores de alta dimensão** onde **proximidade geométrica = similaridade semântica**.

```
"cachorro" → [0.21, -0.54,  0.87, ...]  ┐
"cão"      → [0.19, -0.51,  0.85, ...]  ┘ → vetores próximos (sinônimos)
"carro"    → [-0.63, 0.12, -0.34, ...]    → vetor distante
```

Neste notebook vamos explorar:

1. A matemática — **Similaridade de Cosseno** implementada do zero
2. Gerando **embeddings reais** com `sentence-transformers`
3. Experimento semântico — provando que significado vira distância
4. **Mini busca semântica** — o núcleo de um sistema RAG
5. **Visualização com PCA** — 384 dimensões comprimidas em 2D

> **Dependências:** `sentence-transformers`, `numpy`, `matplotlib`, `scikit-learn` — sem API key necessária.

---

In [4]:
pip install sentence-transformers

  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached torch-2.11.0-cp311-cp311-win_amd64.whl.metadata (29 kB)
Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
Using cached torch-2.11.0-cp311-cp311-win_amd64.whl (114.5 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. A Matemática: Similaridade de Cosseno

Existem várias formas de medir distância entre vetores.  
Em NLP, a mais usada é a **similaridade de cosseno** — ela mede o **ângulo** entre dois vetores, ignorando o tamanho deles.

$$\text{similaridade}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

- Resultado `1.0` → vetores idênticos (ângulo = 0°)
- Resultado `0.0` → vetores perpendiculares (sem relação)
- Resultado `-1.0` → vetores opostos (ângulo = 180°)

Por que não usar distância euclidiana? Porque um texto longo e um texto curto com o mesmo significado teriam vetores de tamanhos diferentes — o cosseno normaliza isso.

In [2]:
import numpy as np

def similaridade_cosseno(a, b):
    """Calcula similaridade de cosseno entre dois vetores."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Exemplo em 2D para visualizar a intuição
v1 = [1, 0]     # aponta para direita
v2 = [1, 0.2]   # quase igual
v3 = [0, 1]     # perpendicular
v4 = [-1, 0]    # oposto

print("Similaridade de Cosseno — exemplos em 2D")
print(f"  v1 vs v2 (quase iguais)   : {similaridade_cosseno(v1, v2):.4f}")
print(f"  v1 vs v3 (perpendiculares): {similaridade_cosseno(v1, v3):.4f}")
print(f"  v1 vs v4 (opostos)        : {similaridade_cosseno(v1, v4):.4f}")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Similaridade de Cosseno — exemplos em 2D
  v1 vs v2 (quase iguais)   : 0.9806
  v1 vs v3 (perpendiculares): 0.0000
  v1 vs v4 (opostos)        : -1.0000


## 2. Gerando Embeddings Reais

Vamos usar o modelo `paraphrase-multilingual-MiniLM-L12-v2` — ele suporta português nativamente e gera vetores de **384 dimensões**.

Na primeira execução ele vai baixar o modelo (~120MB). Nas próximas, usa o cache local.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carrega o modelo multilingual
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Frases de teste
frases = [
    "O cachorro correu pelo parque",
    "O cão foi passear na praça",
    "O gato dormiu no sofá",
]

embeddings = modelo.encode(frases)

print(f"Formato dos embeddings : {embeddings.shape}")
print(f"Frases codificadas     : {len(frases)}")
print(f"Dimensões por embedding: {embeddings.shape[1]}")
print()
print("Primeiros 8 valores do embedding de cada frase:")
for frase, emb in zip(frases, embeddings):
    print(f"  '{frase[:35]}...' → {np.round(emb[:8], 3)}")

ModuleNotFoundError: No module named 'sentence_transformers'

## 3. Experimento Semântico

**Hipótese:** frases com significado similar devem ter similaridade de cosseno maior do que frases com significados diferentes — mesmo usando palavras completamente diferentes.

Vamos testar com grupos temáticos: animais, tecnologia e culinária.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

grupos = {
    "🐾 Animais": [
        "O cachorro latiu para o carteiro",
        "O gato miou pedindo comida",
        "O pássaro cantou ao amanhecer",
    ],
    "💻 Tecnologia": [
        "A rede neural foi treinada por dias",
        "O modelo de linguagem processou milhões de tokens",
        "O algoritmo convergiu após 100 épocas",
    ],
    "🍳 Culinária": [
        "O chef temperou o frango antes de assar",
        "A receita pede duas colheres de azeite",
        "O bolo ficou dourado no forno",
    ]
}

# Gera embeddings de todas as frases
todas_frases = [f for grupo in grupos.values() for f in grupo]
todos_labels = [label for label, grupo in grupos.items() for _ in grupo]
todos_embs   = modelo.encode(todas_frases)

# Calcula similaridade intra-grupo vs inter-grupo
print("=== Similaridades Médias ===")
print()

nomes = list(grupos.keys())
tamanho = len(list(grupos.values())[0])

for i, nome_i in enumerate(nomes):
    embs_i = todos_embs[i*tamanho:(i+1)*tamanho]
    
    for j, nome_j in enumerate(nomes):
        embs_j = todos_embs[j*tamanho:(j+1)*tamanho]
        
        sims = []
        for ei in embs_i:
            for ej in embs_j:
                if not np.array_equal(ei, ej):
                    sims.append(similaridade_cosseno(ei, ej))
        
        tipo = "(intra)" if i == j else "(inter)"
        print(f"  {nome_i} ↔ {nome_j} {tipo}: {np.mean(sims):.4f}")
    print()

**O que observar:**
- Similaridades **intra-grupo** devem ser consistentemente maiores que as **inter-grupo**
- Isso prova que o modelo capturou semântica, não apenas palavras em comum
- Frases sobre animais e culinária, por exemplo, devem ter similaridade muito baixa

## 4. Mini Busca Semântica

Esta é a seção mais importante para o seu roadmap: **busca semântica é o coração de um sistema RAG**.

O fluxo é simples:
1. Você tem um corpus de documentos
2. Gera embeddings de todos eles
3. Recebe uma query do usuário
4. Gera o embedding da query
5. Retorna os documentos com maior similaridade de cosseno

Isso é exatamente o que acontece no passo de **retrieval** de um RAG.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Corpus: simula uma base de documentos
corpus = [
    "Redes neurais são compostas por camadas de neurônios artificiais.",
    "O processo de backpropagation ajusta os pesos durante o treinamento.",
    "Embeddings transformam texto em vetores numéricos de alta dimensão.",
    "A temperatura controla a aleatoriedade na geração de tokens.",
    "RAG combina recuperação de documentos com geração de linguagem.",
    "O Brasil é o maior produtor de café do mundo.",
    "A seleção brasileira ganhou cinco Copas do Mundo.",
    "Python é a linguagem mais usada em ciência de dados.",
    "Docker facilita a criação de ambientes reproduzíveis.",
    "MLflow é uma ferramenta de rastreamento de experimentos de ML.",
]

# Indexa o corpus (em produção, isso é feito uma vez e salvo num vector store)
print("Indexando corpus...")
embeddings_corpus = modelo.encode(corpus)
print(f"Corpus indexado: {len(corpus)} documentos de {embeddings_corpus.shape[1]} dimensões\n")

def busca_semantica(query, top_k=3):
    """Retorna os top_k documentos mais similares à query."""
    emb_query = modelo.encode([query])[0]
    
    similaridades = [
        (i, similaridade_cosseno(emb_query, emb_doc))
        for i, emb_doc in enumerate(embeddings_corpus)
    ]
    
    resultados = sorted(similaridades, key=lambda x: x[1], reverse=True)[:top_k]
    return resultados

# Teste com queries diferentes
queries = [
    "Como funciona o treinamento de modelos de linguagem?",
    "Ferramentas para MLOps e rastreamento de experimentos",
    "Futebol no Brasil",
]

for query in queries:
    print(f"🔍 Query: '{query}'")
    resultados = busca_semantica(query, top_k=3)
    for rank, (idx, sim) in enumerate(resultados, 1):
        print(f"  [{rank}] sim={sim:.4f} → {corpus[idx]}")
    print()

**O que observar:**
- A query sobre treinamento de modelos deve recuperar documentos sobre backpropagation e redes neurais — mesmo sem palavras em comum
- A query sobre futebol deve puxar o documento da seleção brasileira, não os de ML
- Isso funciona **sem nenhuma palavra-chave** — é puramente semântico

> **Conexão com RAG:** num sistema completo, os documentos recuperados aqui seriam passados como contexto para um LLM gerar a resposta final. O retrieval que você acabou de implementar é o `R` do RAG.

## 5. Visualização com PCA

Embeddings têm 384 dimensões — impossível visualizar diretamente.  
**PCA (Análise de Componentes Principais)** comprime esses 384 números em 2, preservando o máximo de variância possível.

O resultado esperado: **frases do mesmo grupo devem formar clusters visuais**.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Frases com grupos temáticos claros
frases_pca = [
    # ML / IA
    "Redes neurais aprendem com dados",
    "O modelo foi treinado por 100 épocas",
    "Embeddings capturam significado semântico",
    "A temperatura afeta a geração de texto",
    "Transformers revolucionaram o NLP",
    # Esportes
    "O time marcou um gol no último minuto",
    "O atleta bateu o recorde mundial",
    "A partida terminou em empate",
    "O técnico mudou a escalação no intervalo",
    "O torneio começa na próxima semana",
    # Culinária
    "A receita leva farinha e ovos",
    "O chef temperou a carne com ervas finas",
    "O bolo precisa assar por 40 minutos",
    "Adicione sal a gosto e mexa bem",
    "O molho ficou encorpado e saboroso",
]

grupos_pca = ["🤖 ML/IA"] * 5 + ["⚽ Esportes"] * 5 + ["🍳 Culinária"] * 5
cores_pca  = ["#1f77b4"] * 5 + ["#2ca02c"] * 5 + ["#d62728"] * 5

# Gera e reduz embeddings
embs_pca = modelo.encode(frases_pca)
pca = PCA(n_components=2)
coords = pca.fit_transform(embs_pca)

variancia = pca.explained_variance_ratio_

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("#f8f9fa")
ax.set_facecolor("#f8f9fa")

for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, color=cores_pca[i], s=120, zorder=3, edgecolors="white", linewidths=1.5)
    ax.annotate(
        frases_pca[i][:30] + ("..." if len(frases_pca[i]) > 30 else ""),
        (x, y), fontsize=8, xytext=(6, 4),
        textcoords="offset points", color="#333333"
    )

# Legenda
patches = [
    mpatches.Patch(color="#1f77b4", label="🤖 ML/IA"),
    mpatches.Patch(color="#2ca02c", label="⚽ Esportes"),
    mpatches.Patch(color="#d62728", label="🍳 Culinária"),
]
ax.legend(handles=patches, fontsize=11, loc="upper right",
          framealpha=0.9, edgecolor="#cccccc")

ax.set_title(
    f"Embeddings visualizados com PCA\n"
    f"Variância explicada: PC1={variancia[0]*100:.1f}% | PC2={variancia[1]*100:.1f}%",
    fontsize=13, fontweight="bold", pad=15
)
ax.set_xlabel(f"Componente Principal 1 ({variancia[0]*100:.1f}%)", fontsize=11)
ax.set_ylabel(f"Componente Principal 2 ({variancia[1]*100:.1f}%)", fontsize=11)
ax.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("embeddings_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo como 'embeddings_pca.png'")

**O que observar no gráfico:**
- Frases do mesmo grupo devem aparecer **agrupadas espacialmente** — mesmo sem você ter dito ao modelo quais são os grupos
- Grupos temáticos muito diferentes (ML vs Culinária) devem estar **mais afastados** que grupos mais relacionados
- PCA perde informação — a separação em 2D é sempre menos nítida que em 384D

## 6. Conexão com os Outros Conceitos

```
Texto
  ↓
Tokenização  → converte texto em IDs numéricos (tiktoken / WordPiece)
  ↓
Embeddings   → converte IDs em vetores com significado semântico
  ↓
Atenção      → o modelo relaciona vetores entre si (Transformer)
  ↓
Logits       → scores brutos para cada token do vocabulário
  ↓
Temperatura  → reshapa a distribuição de probabilidade dos logits
  ↓
Sampling     → escolha do próximo token
```

Você agora entendeu experimentalmente **quatro etapas** dessa cadeia.

---

## 7. Conclusões

**Sobre embeddings:**
- Embeddings convertem texto em vetores onde distância geométrica = similaridade semântica
- Similaridade de cosseno é a métrica padrão — ela ignora magnitude e foca em direção
- Modelos multilinguais como o `MiniLM` funcionam em português sem configuração extra

**Sobre busca semântica:**
- Você implementou do zero o componente de **retrieval** de um sistema RAG
- Em produção, os embeddings do corpus ficam armazenados em um **vector store** (Pinecone, Chroma, FAISS)
- A query é convertida em embedding em tempo real e comparada contra o índice

**Sobre PCA:**
- PCA é uma ferramenta de análise, não de produção
- Clusters visíveis em 2D confirmam que o modelo aprendeu estrutura semântica real
- Para visualizações mais ricas, t-SNE e UMAP preservam melhor estruturas locais

---

### Próximo passo natural: Projeto RAG
Com tokenização, temperatura e embeddings dominados, você tem a base para construir um pipeline RAG completo:
- **Ingestão** de documentos → chunking → embeddings → vector store
- **Retrieval** → busca semântica (o que você implementou aqui)
- **Geração** → LLM recebe os chunks recuperados como contexto e gera a resposta